# Challenge 1 - Tic Tac Toe

In this lab you will perform deep learning analysis on a dataset of playing [Tic Tac Toe](https://en.wikipedia.org/wiki/Tic-tac-toe).

There are 9 grids in Tic Tac Toe that are coded as the following picture shows:

![Tic Tac Toe Grids](tttboard.jpg)

In the first 9 columns of the dataset you can find which marks (`x` or `o`) exist in the grids. If there is no mark in a certain grid, it is labeled as `b`. The last column is `class` which tells you whether Player X (who always moves first in Tic Tac Toe) wins in this configuration. Note that when `class` has the value `False`, it means either Player O wins the game or it ends up as a draw.

Follow the steps suggested below to conduct a neural network analysis using Tensorflow and Keras. You will build a deep learning model to predict whether Player X wins the game or not.

## Step 1: Data Engineering

This dataset is almost in the ready-to-use state so you do not need to worry about missing values and so on. Still, some simple data engineering is needed.

1. Read `tic-tac-toe.csv` into a dataframe.
1. Inspect the dataset. Determine if the dataset is reliable by eyeballing the data.
1. Convert the categorical values to numeric in all columns.
1. Separate the inputs and output.
1. Normalize the input data.

In [50]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

tf.keras.utils.set_random_seed(42)
df = pd.read_csv('tic-tac-toe.csv')

print("--- Data Info ---")
df.info()

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Class Distribution ---")
print(df.iloc[:, -1].value_counts())

missing_count = df.isnull().sum().sum()
print(f"\nTotal Missing Values: {missing_count}")

unique_values = {col: df[col].unique().tolist() for col in df.columns[:-1]}
print("\nUnique values per feature column:")
print(unique_values)

X_raw = df.iloc[:, :-1]
y_raw = df.iloc[:, -1]

X_train_val_raw, X_test_raw, y_train_val_raw, y_test_raw = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)
X_train_raw, X_val_raw, y_train_raw, y_val_raw = train_test_split(
    X_train_val_raw, y_train_val_raw, test_size=0.2, random_state=42,
    stratify=y_train_val_raw
)

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train = encoder.fit_transform(X_train_raw)
X_val = encoder.transform(X_val_raw)
X_test = encoder.transform(X_test_raw)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val = label_encoder.transform(y_val_raw)
y_test = label_encoder.transform(y_test_raw)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set shape:   {X_train_scaled.shape}")
print(f"Validation set shape: {X_val_scaled.shape}")
print(f"Testing set shape:    {X_test_scaled.shape}")
print(f"Class mapping: {dict(enumerate(label_encoder.classes_))}")
print("Inputs encoded and normalized using training data only.")

--- Data Info ---
<class 'pandas.DataFrame'>
RangeIndex: 958 entries, 0 to 957
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   TL      958 non-null    str  
 1   TM      958 non-null    str  
 2   TR      958 non-null    str  
 3   ML      958 non-null    str  
 4   MM      958 non-null    str  
 5   MR      958 non-null    str  
 6   BL      958 non-null    str  
 7   BM      958 non-null    str  
 8   BR      958 non-null    str  
 9   class   958 non-null    bool 
dtypes: bool(1), str(9)
memory usage: 68.4 KB

--- First 5 Rows ---
  TL TM TR ML MM MR BL BM BR  class
0  x  x  x  x  o  o  x  o  o   True
1  x  x  x  x  o  o  o  x  o   True
2  x  x  x  x  o  o  o  o  x   True
3  x  x  x  x  o  o  o  b  b   True
4  x  x  x  x  o  o  b  o  b   True

--- Class Distribution ---
class
True     626
False    332
Name: count, dtype: int64

Total Missing Values: 0

Unique values per feature column:
{'TL': ['x', 'o', 'b'], 'TM': ['x', 'o

## Step 2: Build Neural Network

Create and train a baseline model. Use the validation set to monitor performance and keep the test set untouched until the final model is selected.

1. Create a `Sequential` model.
1. Add several layers. Use ReLU for the hidden layers and Softmax for the output layer.
1. Compile with the `adam` optimizer, `sparse_categorical_crossentropy` loss, and `accuracy` metric.
1. Fit the model using the training set and monitor validation performance.
1. Save the baseline model as `tic-tac-toe-baseline.keras`.

In [51]:
model = Sequential([
    tf.keras.Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(2, activation='softmax')
])

model.summary()

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=16,
    validation_data=(X_val_scaled, y_val),
    verbose=1
)

baseline_val_loss, baseline_val_accuracy = model.evaluate(
    X_val_scaled, y_val, verbose=0
)
print("\n--- Baseline Validation Results ---")
print(f"Validation Loss:     {baseline_val_loss:.4f}")
print(f"Validation Accuracy: {baseline_val_accuracy * 100:.2f}%")

model.save('tic-tac-toe-baseline.keras')
print("\nBaseline model saved as 'tic-tac-toe-baseline.keras'.")

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_29 (Dense)                │ (None, 64)             │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,938 (15.38 KB)

 Trainable params: 3,938 (15.38 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.6013 - loss: 0.6644 - val_accuracy: 0.6883 - val_loss: 0.6090
Epoch 2/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7092 - loss: 0.5665 - val_accuracy: 0.7338 - val_loss: 0.5580
Epoch 3/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7712 - loss: 0.5065 - val_accuracy: 0.7338 - val_loss: 0.5211
Epoch 4/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7810 - loss: 0.4613 - val_accuracy: 0.7597 - val_loss: 0.4981
Epoch 5/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8007 - loss: 0.4389 - val_accuracy: 0.7727 - val_loss: 0.4693
Epoch 6/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8399 - loss: 0.3726 - val_accuracy: 0.8117 - val_loss: 0.4348
Epoch 7/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8497 - loss: 0.3714 - val_accuracy: 0.8117 - val_loss: 0.3963
Epoch 8/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8742 - loss: 0.3217 - val_accuracy: 0.8377 - val_loss

## Step 3: Make Predictions

Load the saved baseline model and make predictions on a few random rows from the validation set. Check whether the predictions are correct. The test set remains reserved for the final evaluation.

In [52]:
import numpy as np
import tensorflow as tf

loaded_model = tf.keras.models.load_model('tic-tac-toe-baseline.keras')
print("Baseline model loaded successfully.\n")

np.random.seed(42)
num_samples = 5
random_indices = np.random.choice(len(X_val_scaled), size=num_samples, replace=False)

sample_inputs = X_val_scaled[random_indices]
true_labels = y_val[random_indices]
label_map = {
    class_index: ("Win / Positive" if bool(label) else "Loss / Negative")
    for class_index, label in enumerate(label_encoder.classes_)
}

raw_predictions = loaded_model.predict(sample_inputs, verbose=0)
predicted_classes = np.argmax(raw_predictions, axis=1)

print("=" * 65)
print(f"{'Sample #':<10} | {'True Label':<18} | {'Predicted':<18} | {'Match?'}")
print("=" * 65)

for i in range(num_samples):
    true_str = label_map[true_labels[i]]
    pred_str = label_map[predicted_classes[i]]
    is_correct = "Correct" if true_labels[i] == predicted_classes[i] else "Incorrect"
    print(f"Sample {i+1:<3} | {true_str:<18} | {pred_str:<18} | {is_correct}")

print("=" * 65)
print("\n--- Raw Prediction Probabilities ---")
for i in range(num_samples):
    p_loss = raw_predictions[i][0] * 100
    p_win = raw_predictions[i][1] * 100
    print(f"Sample {i+1}: Loss Probability = {p_loss:5.2f}% | Win Probability = {p_win:5.2f}%")

Baseline model loaded successfully.

Sample #   | True Label         | Predicted          | Match?
Sample 1   | Win / Positive     | Win / Positive     | Correct
Sample 2   | Win / Positive     | Win / Positive     | Correct
Sample 3   | Win / Positive     | Win / Positive     | Correct
Sample 4   | Win / Positive     | Win / Positive     | Correct
Sample 5   | Win / Positive     | Win / Positive     | Correct

--- Raw Prediction Probabilities ---
Sample 1: Loss Probability =  0.37% | Win Probability = 99.63%
Sample 2: Loss Probability =  0.39% | Win Probability = 99.61%
Sample 3: Loss Probability =  0.00% | Win Probability = 100.00%
Sample 4: Loss Probability =  0.00% | Win Probability = 100.00%
Sample 5: Loss Probability =  0.00% | Win Probability = 100.00%


## Step 4: Compare an Improved Model

The baseline is evaluated on the validation set. Use validation performance to guide experiments and select a model; do not use the test set for tuning. The test set is evaluated once after selection.

Possible experiments include:

* Change the number and size of hidden layers.
* Adjust the learning rate with a custom `tf.keras.optimizers.Adam` instance.
* Train for more epochs while using early stopping on validation loss.
* Try Batch Normalization or Dropout as regularization techniques.

In [53]:
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

custom_optimizer = Adam(learning_rate=0.0005)

improved_model = Sequential([
    tf.keras.Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.15),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.15),
    Dense(32, activation='relu'),
    Dense(2, activation='softmax')
])

improved_model.compile(
    optimizer=custom_optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = improved_model.fit(
    X_train_scaled,
    y_train,
    epochs=120,
    batch_size=16,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stop],
    verbose=1
)

improved_val_loss, improved_val_accuracy = improved_model.evaluate(
    X_val_scaled, y_val, verbose=0
)

print("\n--- Validation Model Comparison ---")
print(f"Baseline loss: {baseline_val_loss:.4f}; accuracy: {baseline_val_accuracy * 100:.2f}%")
print(f"Improved loss: {improved_val_loss:.4f}; accuracy: {improved_val_accuracy * 100:.2f}%")

if improved_val_loss <= baseline_val_loss:
    selected_model = improved_model
    selected_model_name = "improved"
else:
    selected_model = model
    selected_model_name = "baseline"

print(f"Selected model using validation loss: {selected_model_name}")

test_loss, test_accuracy = selected_model.evaluate(X_test_scaled, y_test, verbose=0)
print("\n--- Final Test Results (evaluated once) ---")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Meets targets: {test_loss < 0.1 and test_accuracy > 0.95}")

selected_model.save('tic-tac-toe.keras')
print("\nSelected model saved as 'tic-tac-toe.keras'.")

Epoch 1/120
39/39 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.6078 - loss: 0.7016 - val_accuracy: 0.6429 - val_loss: 0.6389
Epoch 2/120
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7124 - loss: 0.5496 - val_accuracy: 0.6948 - val_loss: 0.5983
Epoch 3/120
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7680 - loss: 0.4720 - val_accuracy: 0.7208 - val_loss: 0.5665
Epoch 4/120
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7843 - loss: 0.4412 - val_accuracy: 0.7273 - val_loss: 0.5375
Epoch 5/120
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8268 - loss: 0.3858 - val_accuracy: 0.7403 - val_loss: 0.5045
Epoch 6/120
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8578 - loss: 0.3415 - val_accuracy: 0.7532 - val_loss: 0.4791
Epoch 7/120
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8627 - loss: 0.3187 - val_accuracy: 0.7792 - val_loss: 0.4483
Epoch 8/120
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8938 - loss: 0.2828 - val_accuracy: 0.8117 - 

**Which approach(es) did you find helpful to improve your model performance?**

The baseline met both validation targets in this run (loss `0.0696`, accuracy `96.75%`). The candidate combining a deeper network, a lower learning rate, Batch Normalization, Dropout, and early stopping scored worse on validation (loss `0.0841`, accuracy `96.10%`), so the baseline was selected. On the untouched test set, the selected model achieved loss `0.0989` and accuracy `96.35%`, meeting both targets.

Because those changes were tried together, this comparison does not identify the effect of any one change. Batch Normalization normalizes activations using batch statistics during training and moving statistics at inference. Dropout randomly disables units during training as a regularization technique; it may help generalization but does not guarantee it.